In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import xgboost
from xgboost import plot_tree, to_graphviz
import holidays as hdays


df = pd.read_excel('caffe_change.xlsx')

In [30]:
holidays = hdays.IN()
df['Is_holiday'] = df['Date'].apply(lambda x: 1 if x in holidays else 0)

In [31]:
df['Date'] = pd.to_datetime(df['Date'])

daily = (df.groupby('Date')
                   .agg({
                       'Quantity': 'sum',
                       'Total': 'sum',
                       'Weekday': 'first',
                       'Week number': 'first',
                       'Is_holiday': 'max'
                   })
                   .reset_index()
                   .rename(columns={'Quantity': 'Total_Units', 'Total': 'Total_Sales'})
)

daily.sort_values(by='Date', inplace=True)

week_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
days = daily["Weekday"].unique()

sorted_days = sorted(days, key=lambda d: week_order.index(d))

weekday_dict = {}
for x in range(len(sorted_days)):
    weekday_dict[sorted_days[x]] = x

daily["Weekday"] = daily["Weekday"].map(weekday_dict)

daily["Day"] = daily["Date"].dt.day
daily["Month"] = daily["Date"].dt.month
daily["Year"] = daily["Date"].dt.year


# daily["Timestamp"] = daily["Date"].apply(lambda x: x.timestamp())   

daily.head(400)

,Date,Total_Units,Total_Sales,Weekday,Week number,Is_holiday,Day,Month,Year
0,2010-04-01,434,77066.51,3,13,0,1,4,2010
1,2010-04-02,489,88423.17,4,13,1,2,4,2010
2,2010-04-03,766,140314.76,5,13,0,3,4,2010
3,2010-04-04,467,86284.74,6,13,0,4,4,2010
4,2010-04-05,385,75674.14,0,14,0,5,4,2010
...,...,...,...,...,...,...,...,...,...
360,2011-03-27,441,90777.48,6,12,0,27,3,2011
361,2011-03-28,401,86109.16,0,13,0,28,3,2011
362,2011-03-29,447,87893.02,1,13,0,29,3,2011
363,2011-03-30,707,141621.87,2,13,0,30,3,2011


In [32]:
X_data, Y_data = daily.drop(columns=["Total_Units", "Date"]), daily["Total_Units"]

In [33]:
x_train, x_test = X_data[:250], X_data[250:]
y_train, y_test = Y_data[:250], Y_data[250:]

In [34]:
xgb = xgboost.XGBRegressor(max_depth=28, n_estimators=150)
xgb.fit(x_train, y_train)
y_pred_xgb = xgb.predict(x_test)

print("XGBoost Model Performance:")
print("R^2 Score:", r2_score(y_test, y_pred_xgb))
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred_xgb))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred_xgb))

XGBoost Model Performance:
R^2 Score: 0.78542160987854
Mean Absolute Error: 38.136573791503906
Mean Squared Error: 2412.78857421875


In [ ]:
# plot actual vs predicted for the test set
dates = daily['Date'].iloc[250:]  # test dates
plt.figure(figsize=(12,5))
plt.plot(dates, y_test.values, label='Actual', marker='o', linestyle='-')
plt.plot(dates, y_pred_xgb, label='Predicted (XGB)', marker='o', linestyle='--')
plt.xlabel('Date')
plt.ylabel('Total Units')
plt.title('Actual vs Predicted Total Units')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()